# Chapter 4: Python's Built-ins Under the Hood

The programs from Chapter 4 of *Data Structures and Algorithms in Python, Hands-On* by Aadhavan Muthurengan, in the book's order. Each code cell is one program; the same program is in this folder as a `.py` file, and its expected output is in the `.out` file with the same name. Benchmark timings differ on every computer and every run.

Run the cells from top to bottom (**Run > Run All Cells**). The programs import each other's files, so keep this notebook in its chapter folder.

## Watching a List Grow

`list_growth.py`

In [1]:
"""Watch a list grow: its size in memory jumps, then stays put."""

import sys

items = []
last = sys.getsizeof(items)
print(f"empty list: {last} bytes")
for i in range(40):
    items.append(i)
    size = sys.getsizeof(items)
    if size != last:
        slots = (size - sys.getsizeof([])) // 8  # 8 bytes per reference
        print(f"length {len(items):>2}: {size:>3} bytes, room for {slots}")
        last = size

empty list: 56 bytes
length  1:  88 bytes, room for 4
length  5: 120 bytes, room for 8
length  9: 184 bytes, room for 16
length 17: 248 bytes, room for 24
length 25: 312 bytes, room for 32
length 33: 376 bytes, room for 40


## What Each List Operation Costs

`bench_list_ops.py`

In [2]:
"""What each list operation costs, as the list grows."""

import bench


def make(n):
    return (list(range(n)),)


def append_pop(items):  # add at the end, then take it off again
    items.append(0)
    items.pop()


def insert_pop_front(items):  # add at the front, then take it off
    items.insert(0, 0)
    items.pop(0)


def read_middle(items):
    return items[len(items) // 2]


sizes = [10_000, 20_000, 40_000, 80_000, 160_000]
results = {
    "append + pop()": bench.doubling(append_pop, make, sizes),
    "items[i]": bench.doubling(read_middle, make, sizes),
    "insert(0) + pop(0)": bench.doubling(insert_pop_front, make, sizes),
}
bench.table(results)
bench.chart(results, "ch04_list_ops.png", "List operations")

         n  append + pop()        items[i]  insert(0) + pop(0)
    10,000         78.7 ns         88.5 ns             2.96 us
    20,000    80.2 ns x1.0    87.5 ns x1.0        6.59 us x2.2
    40,000      79 ns x1.0    87.1 ns x1.0          14 us x2.1
    80,000    78.6 ns x1.0    87.9 ns x1.0        28.8 us x2.1
   160,000    80.9 ns x1.0    85.4 ns x1.0        61.3 us x2.1


## Dictionaries and Sets: Hash Tables

`hashing.py`

In [3]:
"""Hash values: what dicts and sets use to find things."""

print(hash(42), hash(42) == hash(42.0))  # equal values, equal hashes
print(hash(-1), hash(-2))  # -1 is special in CPython: it becomes -2
print(hash((1, 2)) == hash((1, 2)))  # tuples hash by their contents

try:
    hash([1, 2])
except TypeError as error:
    print("TypeError:", error)  # a list can change, so it can't be a key

point = {(0, 0): "origin", (1, 2): "P"}
print(point[(1, 2)])

42 True
-2 -2
True
TypeError: unhashable type: 'list'
P


`bench_dict_set.py`

In [4]:
"""Dict and set operations take about the same time at any size."""

import bench


def make_dict(n):
    return ({i: i for i in range(n)},)


def make_set(n):
    return (set(range(n)),)


def dict_get(d):
    return d.get(12345)


def dict_set_delete(d):
    d[-1] = 0
    del d[-1]


def set_add_remove(s):
    s.add(-1)
    s.remove(-1)


sizes = [10_000, 100_000, 1_000_000]
results = {
    "dict get": bench.doubling(dict_get, make_dict, sizes),
    "dict set + del": bench.doubling(dict_set_delete, make_dict, sizes),
    "set add + remove": bench.doubling(set_add_remove, make_set, sizes),
}
bench.table(results)

         n        dict get  dict set + del  set add + remove
    10,000         73.2 ns         91.7 ns           92.9 ns
   100,000    85.9 ns x1.2    98.6 ns x1.1      95.9 ns x1.0
 1,000,000    86.1 ns x1.0    92.9 ns x0.9      93.8 ns x1.0


## Strings Can't Change

`bench_strings.py`

In [5]:
"""Building a long string: += in a loop or ''.join()?"""

import bench


def make(n):
    return (["word"] * n,)


def plus_equals(words):
    text = ""
    for w in words:
        text += w
    return text


def join(words):
    return "".join(words)


def plus_equals_two_names(words):
    text = ""
    for w in words:
        alias = text  # a second reference to the same string
        text += w
    return alias


sizes = [5_000, 10_000, 20_000, 40_000]
results = {
    "+=": bench.doubling(plus_equals, make, sizes),
    "join": bench.doubling(join, make, sizes),
    "+= (shared)": bench.doubling(plus_equals_two_names, make, sizes),
}
bench.table(results)

         n              +=            join     += (shared)
     5,000          159 us         16.7 us          762 us
    10,000     300 us x1.9    33.4 us x2.0    4.41 ms x5.8
    20,000     577 us x1.9    66.5 us x2.0    19.7 ms x4.5
    40,000    1.12 ms x1.9     132 us x2.0    82.3 ms x4.2
